Место для инсайтов\гипотез\мыслей

1. Взял топ 20 кук по частотности в файле events и получил что бОльшая часть кук являются ботами - можно взять за baseline

2. Сплит трейн\тест: 70:30

3. таргет в трейне имеет явный дисбаланс 8:92 - задача с дисбалансом классов

4. user_agent несет в себе много информации но стоит сильно чистить, также имеет очевидные красные флаги типо
user-agent = 'Scrapy/2.11.0 (+https://scrapy.org)'

5. item_id имеет 30 % пропусков

6. есть предположение что в item_category категории связанные с недвижимость(kvartiry_arenda,kvartiry_prodazha) будут иметь большое кол во ботов

7. признак item_location нужно проверить на мэтчинг, тк бот может переключать часто города прибывания, хотя человек таким не будет заниматься, вообщем, очень важный признак которые может дать много информации


8. seller_type имеет два уник значение приват и про, в идеале, узнать больше информации от бизнеса, что именно значат эти типы, именно в нашей задаче есть вероятность что данный признак не самый информативный, но во время реальной работы больше узнаю про этот признак(возможно товары pro селлеров являются более желаемой добычей для скрапера). Хотя можно посмотреть и на данных датасетах

9. интересно узнать, в какой момент мы фиксируем позицию мышки на экране, наверно в момент прокидывывания события в кликстрим

10. можно написать функцию через которую будем предобрабатывать позицию мышки, что я имею в виду, большая часть экранов имеют разное расширение, но есть большие экраны, которые будут вносить диссонанс, исходя из этого можно будет как то масштабировать позиции по x\y исходя из размера экрана(идея на будущее)

11. Глубина поиска search_page у бота будет кратно глубже

12. Есть предположение что событийная активность у ботов сильно отличается, например, используя timestamp events_ts мы можем посчитать, сколько времени проходить между каждым событием у куки, вероятнее всего у ботов время между событиями будет сильно меньше, чем у человека, обычный человек намного медленее совершает действия во время серфинга на сайте

In [6]:
import pandas as pd

In [7]:
df_train = pd.read_csv('train.csv')

In [8]:
df_train

,cookie_id,cookie_created_at,window_start_ts,window_end_ts,target
0,ck_54a059eb7d3ea68b,2025-11-21 09:30:41,2026-04-06,2026-04-07,0
1,ck_7e4de46eeab82974,2025-09-23 10:10:24,2026-04-06,2026-04-07,0
2,ck_9320229ef6304522,2026-03-04 00:08:02,2026-04-06,2026-04-07,0
3,ck_30ccd25bc1714ed9,2026-04-05 10:52:40,2026-04-06,2026-04-07,0
4,ck_a77c5f05948cdeef,2026-01-01 03:54:35,2026-04-06,2026-04-07,0
...,...,...,...,...,...
11086,ck_2256c19bb847dd0e,2026-04-18 03:59:35,2026-04-19,2026-04-20,0
11087,ck_ba2e403201cba06d,2026-03-11 01:30:00,2026-04-19,2026-04-20,0
11088,ck_2f43054a1f52b9a2,2026-03-19 03:44:55,2026-04-19,2026-04-20,0
11089,ck_354f57a870e41ddd,2026-04-17 22:44:59,2026-04-19,2026-04-20,0


In [9]:
df_train['target'].value_counts()

target
0    10192
1      899
Name: count, dtype: int64

In [10]:
df_train.target.mean().round(4)

np.float64(0.0811)

In [11]:
899 / (899 + 10192)

0.0810567126498963

### Таргет в train в пропорции 92:8

In [12]:
df_test = pd.read_csv('test.csv')

In [13]:
df_test

,cookie_id,cookie_created_at,window_start_ts,window_end_ts
0,ck_315fb710a0e371e7,2026-02-20 08:52:31,2026-04-20,2026-04-21
1,ck_a76ee3b3e3e522fd,2026-01-19 13:42:15,2026-04-20,2026-04-21
2,ck_94c9a4d382689e82,2026-04-19 06:28:37,2026-04-20,2026-04-21
3,ck_8eaf9509ad9462a0,2026-01-19 17:18:06,2026-04-20,2026-04-21
4,ck_9a88a5a989cb5bc6,2025-11-09 17:25:13,2026-04-20,2026-04-21
...,...,...,...,...
4904,ck_8b772caed8f1541c,2026-03-06 06:17:32,2026-04-26,2026-04-27
4905,ck_5403c76fe68a08ba,2025-09-16 01:17:03,2026-04-26,2026-04-27
4906,ck_3e6833574edf1736,2026-02-06 19:45:02,2026-04-26,2026-04-27
4907,ck_e527fbb9450a4581,2026-04-25 17:28:41,2026-04-26,2026-04-27


In [15]:
df_events = pd.read_csv('events.csv')

In [16]:
df_events.head()

,cookie_id,event_ts,eid,event_name,platform,user_agent,item_id,item_category,item_location,seller_type,search_query,search_page,pointer_x,pointer_y
0,ck_5efbea1befdefe1b,2026-04-26 09:11:24,200,item_view,desktop,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,1027450.0,elektronika,kaliningrad,pro,NaN,NaN,NaN,NaN
1,ck_c4ca1434f3778f1d,2026-04-20 14:04:31,100,search_results_view,web,Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:1...,NaN,telefony,habarovsk,NaN,iphone 13 128,4.0,NaN,NaN
2,ck_d274382b19488771,2026-04-13 12:02:56,200,item_view,WEB,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,1048743.0,kvartiry_prodazha,kirov,private,NaN,NaN,675.0,276.0
3,ck_fbbed2ff14944ce9,2026-04-08 06:12:14,100,search_results_view,web,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,NaN,bytovaya_tehnika,novosibirsk,NaN,пылесос dyson,2.0,NaN,NaN
4,ck_56cc15c7c634cb9f,2026-04-12 17:16:05,100,search_results_view,WEB,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,NaN,odezhda,sankt-peterburg,NaN,костюм мужской,2.0,NaN,NaN


In [17]:
df_events.dtypes

cookie_id         object
event_ts          object
eid                int64
event_name        object
platform          object
user_agent        object
item_id          float64
item_category     object
item_location     object
seller_type       object
search_query      object
search_page      float64
pointer_x        float64
pointer_y        float64
dtype: object

In [18]:
df_events.shape

(328905, 14)

In [19]:
hundread_most_usable_cookies = df_events['cookie_id'].value_counts()[:100].index

In [217]:
len(df_events['cookie_id'].unique())

16000

In [21]:
4909 + 11091

16000

In [22]:
11091 / (11091 + 4909)

0.6931875

preprocessing

In [218]:
df_events['platform'] = df_events['platform'].str.lower()

In [219]:
df_events['platform'].value_counts()

platform
web        138792
android    126875
desktop     46866
ios         12269
iphone       4103
Name: count, dtype: int64

In [246]:
platform_w_cookie_id = df_events[['cookie_id', 'platform']]

In [247]:
platform_target = pd.merge(df_train, platform_w_cookie_id, on='cookie_id')[['target', 'platform']]

In [249]:
platform_target[platform_target['target'] == 1]['platform'].value_counts()

platform
web        24329
android    12391
desktop     8200
ios          863
iphone       297
Name: count, dtype: int64

In [250]:
platform_target[platform_target['target'] == 0]['platform'].value_counts()

platform
android    80219
web        76121
desktop    25753
ios         8290
iphone      2752
Name: count, dtype: int64

Боты наиболее активны на платформе веб

In [24]:
with_scrapy_user_agents = df_events[df_events['user_agent'] == 'Scrapy/2.11.0 (+https://scrapy.org)']['cookie_id'].value_counts().index

In [25]:
for i in with_scrapy_user_agents:
  display(df_train[df_train['cookie_id'] == f"{i}"])

,cookie_id,cookie_created_at,window_start_ts,window_end_ts,target
8036,ck_2074e2a6d6e7e54f,2026-04-09 05:36:08,2026-04-15,2026-04-16,1


,cookie_id,cookie_created_at,window_start_ts,window_end_ts,target
9579,ck_3c9a5960f13eb7e4,2026-04-16 15:17:30,2026-04-17,2026-04-18,1


,cookie_id,cookie_created_at,window_start_ts,window_end_ts,target
654,ck_d0b1ca106b76a57c,2026-04-05 13:06:16,2026-04-06,2026-04-07,1


,cookie_id,cookie_created_at,window_start_ts,window_end_ts,target
3322,ck_6fa4993b489312d9,2025-10-05 14:42:10,2026-04-10,2026-04-11,0


,cookie_id,cookie_created_at,window_start_ts,window_end_ts,target
10791,ck_6a0a78f19df5ad9a,2026-03-25 19:05:59,2026-04-19,2026-04-20,1


...

,cookie_id,cookie_created_at,window_start_ts,window_end_ts,target
9133,ck_0512303bf95c7d36,2026-03-19 03:57:26,2026-04-16,2026-04-17,0


,cookie_id,cookie_created_at,window_start_ts,window_end_ts,target


,cookie_id,cookie_created_at,window_start_ts,window_end_ts,target


,cookie_id,cookie_created_at,window_start_ts,window_end_ts,target
1059,ck_31f761a9c2421a01,2026-01-15 17:44:31,2026-04-07,2026-04-08,0


,cookie_id,cookie_created_at,window_start_ts,window_end_ts,target


In [26]:
df_events['user_agent'].unique()

array(['Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36',
       'Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:113.0) Gecko/20100101 Firefox/113.0',
       'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/113.0.0.0 YaBrowser/23.5.0.0 Safari/537.36',
       'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36',
       'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36',
       'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36',
       'Avito/122.0 (Android 11; M2101K6G) okhttp/4.11.0',
       'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 YaBrowser/23.3.0.0 Safari/537.36',
       'Mozilla/5.0 (iPhone; CPU iPhone OS 15_2 like Mac

In [202]:
user_agent_w_cookie_id = df_events[['cookie_id', 'user_agent']]

In [203]:
user_agent_target = pd.merge(df_train, user_agent_w_cookie_id, on='cookie_id')[['target', 'user_agent']]

In [215]:
user_agent_target[event_name_target['target'] == 1]['user_agent'].value_counts().head(5)

user_agent
Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) HeadlessChrome/118.0.0.0 Safari/537.36            1750
Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) HeadlessChrome/114.0.0.0 Safari/537.36            1466
Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) HeadlessChrome/120.0.0.0 Safari/537.36            1191
Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36    1036
Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:116.0) Gecko/20100101 Firefox/116.0                                          882
Name: count, dtype: int64

In [213]:
user_agent_target[event_name_target['target'] == 0]['user_agent'].value_counts().head(5)

user_agent
Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 YaBrowser/23.0.0.0 Safari/537.36    2337
Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36                 2134
Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36                       2120
Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36                 2119
Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 YaBrowser/23.1.0.0 Safari/537.36    2101
Name: count, dtype: int64

### У ботов заметно различие в операционной системе, первые три позиции user_agent занимают клиенты на линуксе, большая часть ботов раскатаны на линуксе а не на windows 

In [27]:
df_events['event_name'].value_counts()

event_name
item_view               120817
search_results_view     100402
photo_swipe              36517
favorite_add             19049
seller_page_view         17403
contact_phone_show       11316
captcha_shown             7928
login                     6267
contact_chat_open         6125
contact_message_sent      3081
Name: count, dtype: int64

In [186]:
event_name_w_cookie_id = df_events[['cookie_id', 'event_name']]

In [194]:
event_name_target = pd.merge(df_train, event_name_w_cookie_id, on='cookie_id')[['target', 'event_name']]

In [199]:
event_name_target[event_name_target['target'] == 1]['event_name'].value_counts()

event_name
item_view               16776
search_results_view     13647
captcha_shown            6970
photo_swipe              2846
seller_page_view         2319
favorite_add             1142
contact_phone_show       1103
contact_chat_open         625
login                     345
contact_message_sent      307
Name: count, dtype: int64

In [200]:
event_name_target[event_name_target['target'] == 0]['event_name'].value_counts()

event_name
item_view               70293
search_results_view     58674
photo_swipe             23538
favorite_add            12648
seller_page_view        10230
contact_phone_show       6971
login                    4161
contact_chat_open        3732
contact_message_sent     1930
captcha_shown             958
Name: count, dtype: int64

### Боты явно выделяются поведением на сайте, это показывают события.
#### Они кратно чаще видят капчу(стоит узнать лучше, какая механика показа капчи у авито)
#### Реже добавляют в избранное
#### Реже отправляют сообщения 

In [28]:
df_events['item_id'].isnull().sum()

np.int64(114597)

In [161]:
df_events['item_category'].value_counts()

item_category
avtomobili           20822
mebel                20541
bytovaya_tehnika     20161
hobbi                20020
elektronika          20002
uslugi               19869
odezhda              19846
kvartiry_arenda      19713
kvartiry_prodazha    19709
noutbuki             19583
detskie_tovary       19441
rabota               19387
telefony             19349
zhivotnye            19109
zapchasti            18433
Name: count, dtype: int64

Со всем категориями пользовали взаимодействуют равномерно(у всех схожее количество событий), что странно, но наверно организаторы дали наиболее репрезентативный датасет

In [163]:
item_category_w_cookie_id = df_events[['cookie_id', 'item_category']]

In [166]:
item_category_target = pd.merge(df_train, item_category_w_cookie_id, on='cookie_id')[['target', 'item_category']]

In [170]:
item_category_target[item_category_target['target'] == 1]['item_category'].value_counts()

item_category
odezhda              2810
bytovaya_tehnika     2768
uslugi               2767
avtomobili           2687
rabota               2598
elektronika          2568
kvartiry_arenda      2475
mebel                2460
detskie_tovary       2388
telefony             2386
hobbi                2356
kvartiry_prodazha    2337
zhivotnye            2123
noutbuki             1946
zapchasti            1806
Name: count, dtype: int64

In [171]:
item_category_target[item_category_target['target'] == 0]['item_category'].value_counts()

item_category
avtomobili           12523
mebel                12280
noutbuki             12213
kvartiry_prodazha    12153
hobbi                12086
odezhda              11819
detskie_tovary       11731
telefony             11720
elektronika          11640
zapchasti            11609
uslugi               11609
kvartiry_arenda      11475
bytovaya_tehnika     11353
zhivotnye            11231
rabota               11229
Name: count, dtype: int64

### Категории товаров не выделяются у ботов - нет перекоса в сторону недвижимости или авто. В топе одежда и бытовая техника

In [128]:
df_events['item_location'].value_counts().head(10)

item_location
sankt-peterburg    39221
novosibirsk        37989
moskva             37964
ekaterinburg       37281
ryazan              4627
izhevsk             4576
barnaul             4559
astrahan            4511
ufa                 4481
vladivostok         4469
Name: count, dtype: int64

In [101]:
item_location_w_cookie_id = df_events[['cookie_id', 'item_location']]

In [105]:
item_location_target = pd.merge(df_train, item_location_w_cookie_id, on='cookie_id')[['cookie_id', 'target', 'item_location']]

In [117]:
item_location_bot_freq = item_location_target[item_location_target['target'] == 1]['item_location'].value_counts().reset_index()

In [118]:
item_location_human_freq = item_location_target[item_location_target['target'] == 0]['item_location'].value_counts().reset_index()

In [129]:
item_location_human_freq['place'] = item_location_human_freq.index + 1

In [122]:
item_location_bot_freq['place'] = item_location_bot_freq.index + 1

In [149]:
human_loc_item_places = item_location_human_freq.sort_values('item_location')

In [150]:
bot_loc_item_places = item_location_bot_freq.sort_values('item_location')

In [152]:
concated_sorted = pd.concat([human_loc_item_places,bot_loc_item_places], axis=1)

In [162]:
concated_item_loc

,item_location,count,place,item_location,count,place,diff
0,sankt-peterburg,24176,1,moskva,4732,1,0
1,novosibirsk,22684,2,novosibirsk,4713,2,0
2,ekaterinburg,22655,3,sankt-peterburg,4585,3,0
3,moskva,22432,4,ekaterinburg,4033,4,0
4,ryazan,2800,5,barnaul,708,5,0
5,astrahan,2709,6,ufa,677,6,0
6,rostov-na-donu,2698,7,ryazan,638,7,0
7,kazan,2683,8,volgograd,634,8,0
8,saratov,2667,9,kirov,619,9,0
9,izhevsk,2648,10,ulyanovsk,615,10,0


In [140]:
concated_item_loc = pd.concat([item_location_human_freq,item_location_bot_freq], axis=1)

In [144]:
concated_item_loc['diff'] = human_loc_item_places - bot_loc_item_places

In [145]:
concated_item_loc

,item_location,count,place,item_location,count,place,diff
0,sankt-peterburg,24176,1,moskva,4732,1,0
1,novosibirsk,22684,2,novosibirsk,4713,2,0
2,ekaterinburg,22655,3,sankt-peterburg,4585,3,0
3,moskva,22432,4,ekaterinburg,4033,4,0
4,ryazan,2800,5,barnaul,708,5,0
5,astrahan,2709,6,ufa,677,6,0
6,rostov-na-donu,2698,7,ryazan,638,7,0
7,kazan,2683,8,volgograd,634,8,0
8,saratov,2667,9,kirov,619,9,0
9,izhevsk,2648,10,ulyanovsk,615,10,0


In [130]:
item_location_human_freq

,item_location,count,place
0,sankt-peterburg,24176,1
1,novosibirsk,22684,2
2,ekaterinburg,22655,3
3,moskva,22432,4
4,ryazan,2800,5
5,astrahan,2709,6
6,rostov-na-donu,2698,7
7,kazan,2683,8
8,saratov,2667,9
9,izhevsk,2648,10


Множественная арифметика для выделения уникального города

In [109]:
item_location_bot = item_location_target[item_location_target['target'] == 1]['item_location'].unique()

In [110]:
item_location_human = item_location_target[item_location_target['target'] == 0]['item_location'].unique()

In [112]:
set(item_location_human) - set(item_location_bot) 

set()

Нет города который не используется скраперами, людям интересны все локации нашей страны :)
Частотный анализ тоже не дал 


In [31]:
df_events['seller_type'].value_counts()

seller_type
private    112631
pro         82421
Name: count, dtype: int64

In [32]:
df_events['search_query'].value_counts().index.to_list()

['toyota camry',
 'кофемашина',
 'клининг',
 'renault logan',
 'kia rio 2019',
 'посудомоечная машина',
 'вторичка 2к',
 'шкаф купе',
 'телевизор 55',
 'стол письменный',
 'квартира без комиссии',
 'volkswagen polo',
 'коньки',
 'джинсы levis',
 'skoda octavia',
 'работа курьером',
 'новостройка',
 'xiaomi redmi note 12',
 'samsung a54',
 'пуховик',
 'пальто женское',
 'самокат детский',
 'квартира посуточно',
 'работа грузчиком',
 'iphone 13 128',
 'палатка туристическая',
 'работа водителем',
 'мультиварка',
 'наушники airpods',
 'квадрокоптер',
 'мастер на все руки',
 'детская кроватка',
 'снять 1к квартиру',
 'корм для собак',
 'комод',
 'платье вечернее',
 'репетитор математика',
 'гитара акустическая',
 'микроволновка',
 'стеллаж',
 'iphone 15 pro',
 'велосипед горный',
 'dell xps',
 'гантели',
 'playstation 5',
 'сборка мебели',
 'asus rog',
 'hp pavilion',
 'аренда комнаты',
 'смена 12 часов',
 'кровать 160х200',
 'пылесос dyson',
 'hyundai solaris',
 'квартира от собственника'

In [63]:
search_query_w_cookie_id = df_events[['cookie_id', 'search_query']]

In [104]:
search_query_target = pd.merge(df_train, search_query_w_cookie_id, on='cookie_id')[['cookie_id', 'target', 'search_query']]

In [98]:
search_query_target[search_query_target['target'] == 1]['search_query'].dropna().value_counts().index.to_list()

['грузоперевозки',
 'джинсы levis',
 'микроволновка',
 'hyundai solaris',
 'пуховик',
 'работа грузчиком',
 'снять 1к квартиру',
 'самокат взрослый',
 'наушники airpods',
 'новостройка',
 'смена 12 часов',
 'посудомоечная машина',
 'xiaomi redmi note 12',
 'телевизор 55',
 'удаленная работа',
 'skoda octavia',
 'квартира без комиссии',
 'автокресло',
 'квадрокоптер',
 'комод',
 'аренда комнаты',
 'кровать 160х200',
 'мультиварка',
 'стол письменный',
 'кофемашина',
 'работа водителем',
 'квартира от собственника',
 'видеокарта rtx',
 'кресло компьютерное',
 'гитара акустическая',
 'volkswagen polo',
 'маникюр',
 'костюм мужской',
 'апартаменты аренда',
 'kia rio 2019',
 'hp pavilion',
 'духовой шкаф',
 'электрик',
 'playstation 5',
 'платье вечернее',
 'телефон бу',
 'сборка мебели',
 'детская кроватка',
 'ботинки мужские',
 'клетка для попугая',
 'renault logan',
 'стульчик для кормления',
 'палатка туристическая',
 'самокат детский',
 'клининг',
 'samsung a54',
 'пылесос dyson',
 'по

In [100]:
search_query_target[search_query_target['target'] == 0]['search_query'].dropna().value_counts().index.to_list()

['шкаф купе',
 'toyota camry',
 'вторичка 2к',
 'bmw x5',
 'стеллаж',
 'asus rog',
 'самокат детский',
 'iphone 15 pro',
 'renault logan',
 'коньки',
 'студия купить',
 'работа курьером',
 'ноутбук игровой',
 'volkswagen polo',
 'джинсы levis',
 'пальто женское',
 'kia rio 2019',
 'велосипед горный',
 'iphone 13 128',
 'samsung a54',
 'апартаменты купить',
 'dell xps',
 'детская кроватка',
 'шины зимние',
 'xiaomi redmi note 12',
 'клининг',
 'палатка туристическая',
 'конструктор lego',
 'квартира от собственника',
 'мастер на все руки',
 'работа водителем',
 'кофемашина',
 'квартира посуточно',
 'бампер передний',
 'беговая дорожка',
 'смена 12 часов',
 'кроссовки nike',
 'skoda octavia',
 'платье вечернее',
 'стартер',
 'репетитор математика',
 'новостройка',
 'купить 1к квартиру',
 'масло моторное',
 'кухонный гарнитур',
 'манеж',
 'электрик',
 'квартира с ремонтом',
 'ремонт квартир',
 'диван угловой',
 'квартира без комиссии',
 'hp pavilion',
 'apple watch',
 'диски r16',
 'куртк

Анализ поисковых запросов в разрезе таргета не дал какого то результата, возможно здесь следует произвести более глубокую работу с точки зрения nlp
Как вариант, построить векторные эмбединги запросов от бота и не бота, и в данном пространстве мы увидим какую то гиперплоскость которая разделяет запросы от ботов и людей

Но все таки, думаю что поисковые запросы от ботов и людей имеют схожу природу и информативный признак здесь не найти

Идея - смотреть на так называемую разносторонность(вспомнить термин из recsys) запросов, вероятнее всего отдельно рассматриваемый бот будет искать карточки из одной категории(например, парсер объявлений о квартирах)

In [33]:
df_events['search_page'].value_counts()

search_page
1.0     38909
2.0     22290
3.0     13374
4.0      8483
5.0      5422
6.0      3523
7.0      2350
8.0      1569
9.0      1141
10.0      778
11.0      569
12.0      424
13.0      329
14.0      248
15.0      207
16.0      148
17.0      112
18.0      105
19.0       73
20.0       58
21.0       54
22.0       38
23.0       33
25.0       20
24.0       19
26.0       17
28.0       15
29.0       14
30.0       11
31.0       11
35.0        9
27.0        9
32.0        8
33.0        8
34.0        5
36.0        4
45.0        3
48.0        2
40.0        2
37.0        2
39.0        1
65.0        1
41.0        1
38.0        1
63.0        1
42.0        1
Name: count, dtype: int64

In [34]:
df_events[df_events[['pointer_x']].notnull()]['pointer_x'].value_counts()

pointer_x
0.0       522
634.0      92
334.0      91
639.0      88
617.0      87
         ... 
1794.0     32
1773.0     31
86.0       28
1604.0     26
1920.0     11
Name: count, Length: 1921, dtype: int64

смотрим мин и макс значения на window_start_ts	window_end_ts чтобы понять временные окна в которых находятся train и test

In [35]:
df_test['window_start_ts'].min()

'2026-04-20'

In [36]:
df_test['window_end_ts'].max()

'2026-04-27'

In [37]:
df_train['window_start_ts'].min()

'2026-04-06'

In [38]:
df_train['window_end_ts'].max()

'2026-04-20'

In [62]:
df_events.isnull().mean() * 100

cookie_id         0.000000
event_ts          0.000000
eid               0.000000
event_name        0.000000
platform          0.000000
user_agent        0.000000
item_id          34.841976
item_category    10.008969
item_location     7.234004
seller_type      40.696554
search_query     69.473860
search_page      69.473860
pointer_x        66.999590
pointer_y        66.999590
dtype: float64

Основные атрибуты 

In [40]:
df_events[['eid','event_name']]

,eid,event_name
0,200,item_view
1,100,search_results_view
2,200,item_view
3,100,search_results_view
4,100,search_results_view
...,...,...
328900,210,photo_swipe
328901,100,search_results_view
328902,200,item_view
328903,100,search_results_view


In [41]:
search_page_w_cookie = df_events[['cookie_id','search_page']]

In [42]:
search_page_target = pd.merge(df_train, search_page_w_cookie, on='cookie_id')

In [43]:
df_corr_sp_target = search_page_target[['cookie_id', 'search_page', 'target']]

In [44]:
df_corr_sp_target[df_corr_sp_target['target'] == 1]['search_page'].describe()

count    13647.000000
mean         4.385140
std          4.570099
min          1.000000
25%          1.000000
50%          3.000000
75%          6.000000
max         65.000000
Name: search_page, dtype: float64

In [45]:
df_corr_sp_target[df_corr_sp_target['target'] == 0]['search_page'].describe()

count    58674.000000
mean         2.538978
std          2.147682
min          1.000000
25%          1.000000
50%          2.000000
75%          3.000000
max         33.000000
Name: search_page, dtype: float64

In [54]:
df_corr_sp_target[df_corr_sp_target['target'] == 0]['search_page'].quantile(0.90)

np.float64(5.0)

In [55]:
df_corr_sp_target[df_corr_sp_target['target'] == 1]['search_page'].quantile(0.90)

np.float64(10.0)

#### Среднее значение у отрицательного класса заметно меньше чем у положительного, что говорит о смещение в правую сторону у ботов, 90 квантиль в два раза больше у таргета

In [223]:
df_events.duplicated().sum()

np.int64(4868)

In [226]:
df_train['window_start_ts'].max()

'2026-04-19'

In [228]:
df_events['pointer_x']

0           NaN
1           NaN
2         675.0
3           NaN
4           NaN
          ...  
328900      NaN
328901      NaN
328902      NaN
328903      NaN
328904      NaN
Name: pointer_x, Length: 328905, dtype: float64

In [229]:
pointer_x_w_cookie_id = df_events[['cookie_id', 'pointer_x']]

In [230]:
pointer_y_w_cookie_id = df_events[['cookie_id', 'pointer_y']]

In [231]:
pointer_x_target = pd.merge(df_train, pointer_x_w_cookie_id, on='cookie_id')[['cookie_id', 'target', 'pointer_x']]

In [232]:
pointer_y_target = pd.merge(df_train, pointer_y_w_cookie_id, on='cookie_id')[['cookie_id', 'target', 'pointer_y']]

In [243]:
pointer_x_target[pointer_x_target['target'] == 1]['pointer_x'].dropna()

298       329.0
299       251.0
300         0.0
301       140.0
302       624.0
          ...  
237434    604.0
237435    391.0
237436    604.0
237437    886.0
237438    290.0
Name: pointer_x, Length: 11801, dtype: float64

In [238]:
pointer_x_target[pointer_x_target['target'] == 0]['pointer_x'].dropna().describe()

count    67332.000000
mean       951.685647
std        553.276202
min          0.000000
25%        474.000000
50%        946.000000
75%       1432.000000
max       1920.000000
Name: pointer_x, dtype: float64

In [240]:
pointer_y_target[pointer_y_target['target'] == 1]['pointer_y'].dropna().describe()

count    11801.000000
mean       500.918058
std        263.142339
min          0.000000
25%        317.000000
50%        494.000000
75%        687.000000
max       1080.000000
Name: pointer_y, dtype: float64

In [245]:
pointer_y_target[pointer_y_target['target'] == 0]['pointer_y'].dropna().describe()

count    67332.000000
mean       541.231198
std        309.866238
min          0.000000
25%        276.000000
50%        542.000000
75%        807.000000
max       1080.000000
Name: pointer_y, dtype: float64

По описательным статистикам тяжело сделать какие то выводы, я нашел статью про детекцию ботов на основании движения курсора мыши, поделюсь ей в гите